# 13 · HDA: Landsat discovery and controlled retrieval

**Objective:** Discover Landsat Collection 2 Level-2 items in the DestinE Data Lake and select either one direct asset or one resumable order.

**Route:** Live API / DestinE HDA authentication · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a discovery table and optionally one downloaded asset or one order record. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/data-lake-hda.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Authenticate for HDA

The provider `destinepyauth` library includes the `hda` service and its token exchange. Complete DestinE account and provider access prerequisites first. No password or token is saved in this notebook. An expired token requires reauthentication. This notebook uses **DEDL HDA STAC v2**, which is different from EDEN's HDA-branded adapters.


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
from destinepyauth import get_token
from urllib.parse import urlsplit
from examples.hda import request_json, download_asset, submit_order_once, safe_filename

auth = get_token("hda")
token = auth.access_token
del auth
HDA = "https://hda.data.destination-earth.eu"
host = urlsplit(HDA).hostname
STAC = f"{HDA}/stac/v2"
COLLECTION = "EO.NASA.DAT.LANDSAT.C2_L2"
collection = request_json("GET", f"{STAC}/collections/{COLLECTION}", token, host)
print({"id": collection.get("id"), "title": collection.get("title"), "extent": collection.get("extent")})


## 3. Discover parameters and search

Read collection queryables before adding optional collection-specific filters. The minimal request uses bbox/time and ten results; follow pagination in a complete study.


In [ ]:
queryables = request_json("GET", f"{STAC}/collections/{COLLECTION}/queryables", token, host)
print({"queryable_names": list(queryables.get("properties", {}))})
search = {"collections": [COLLECTION], "bbox": [24.55, 60.15, 24.75, 60.30], "datetime": "2024-06-01T00:00:00Z/2024-08-31T23:59:59Z", "limit": 10, "sortby": [{"field": "datetime", "direction": "desc"}]}
result = request_json("POST", f"{STAC}/search", token, host, json=search)
items = result.get("features", [])
if not items:
    raise ValueError("No items found; broaden dates or check collection access.")
pd.DataFrame([{"id": item["id"], "time": item.get("properties", {}).get("datetime"), "cloud": item.get("properties", {}).get("eo:cloud_cover"), "asset_keys": list(item.get("assets", {}))} for item in items])


## 4. Select a documented data asset

Inspect asset names/roles/media types without printing signed URLs. Set `HDA_ASSET_KEY` to an asset representing the intended data, not an arbitrary first asset/thumbnail. Add only provider-verified hostnames to `HDA_ALLOWED_ASSET_HOSTS` (comma-separated). Redirect destinations must also be explicitly allowed. DEDL authorization headers are sent only to the exact HDA hostname. Choose a small asset when possible; the demonstration caps downloads at 2 GB.


In [ ]:
selected = items[0]
assets = selected.get("assets", {})
asset_table = pd.DataFrame([{"key": key, "title": asset.get("title"), "type": asset.get("type"), "roles": asset.get("roles"), "host": urlsplit(asset.get("href", "")).hostname} for key, asset in assets.items()])
asset_table


In [ ]:
asset_key = os.getenv("HDA_ASSET_KEY", "").strip()
asset = assets.get(asset_key) if asset_key else None
retrieve = next((link for link in selected.get("links", []) if link.get("rel") == "retrieve"), None)
DOWNLOAD = os.getenv("HDA_DOWNLOAD", "false").lower() == "true"
PLACE_ORDER = os.getenv("HDA_PLACE_ORDER", "false").lower() == "true"
if DOWNLOAD and PLACE_ORDER:
    raise ValueError("Choose direct download OR ordering; avoid duplicate retrieval.")
allowed_hosts = {item.strip() for item in os.getenv("HDA_ALLOWED_ASSET_HOSTS", "").split(",") if item.strip()}
downloaded = None
order = None
if DOWNLOAD:
    if not asset or not asset.get("href"):
        raise ValueError("Set HDA_ASSET_KEY to a documented downloadable data asset.")
    downloaded = download_asset(asset["href"], ROOT / "data" / "landsat", safe_filename(f"{selected['id']}_{asset_key}.bin"), allowed_hosts, token=token, authenticated_host=host)
    print({"downloaded_file": downloaded.name, "bytes": downloaded.stat().st_size})
elif PLACE_ORDER:
    if retrieve is None:
        raise ValueError("This item has no documented retrieve link.")
    record = ROOT / "outputs" / (safe_filename(selected["id"]) + "-hda-order.json")
    order = submit_order_once(retrieve, token, host, record)
    print({"order_id": order.get("id") or order.get("order_id"), "status": order.get("status")})
else:
    print("Discovery complete. Enable one retrieval branch only after selecting the asset/order.")


## 5. Inspect an existing order once

If submission is interrupted, inspect your HDA dashboard using the persisted record; do not submit again automatically. The generic HDA response may expose a status/self link. Exact status/download schema varies by backend: consult the current API documentation. This cell performs one status check and does not repeatedly poll or create another order.


In [ ]:
if order:
    status_link = next((link for link in order.get("links", []) if link.get("rel") in {"status", "self"} and link.get("href")), None)
    if status_link:
        status = request_json("GET", status_link["href"], token, host)
        print({"status": status.get("status"), "id": status.get("id")})
    else:
        print("No status link in this schema; use the provider dashboard/API documentation.")
record_provenance("hda-landsat.json", service="DEDL HDA STAC v2", collection=COLLECTION, search=search, selected_item=selected["id"], selected_asset_key=asset_key or None, retrieval_branch="direct" if DOWNLOAD else "order" if PLACE_ORDER else "discovery only", local_file=downloaded.name if downloaded else None, pagination_followed=False)


## Interpret and verify

This retains the original access-only scope; it does not calculate Landsat NDVI. Landsat Collection 2 Level-2 processing needs its own scale/offset and QA masks before spectral analysis. File extensions are not trusted; identify the media type before opening/extracting an archive. Signed asset URLs and full order responses are deliberately omitted from saved outputs.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [DEDL data-access interfaces](https://destine-data-lake-docs.data.destination-earth.eu/en/latest/interfaces/Data-Access/Data-Access.html)
- [HDA API documentation](https://hda.data.destination-earth.eu/docs)
- [Provider authentication and token exchange](https://github.com/SercoSPA/DestinE-Platform-AuthN)
- [USGS Landsat Collection 2 scale factors](https://www.usgs.gov/faqs/how-do-i-use-a-scale-factor-landsat-level-2-science-products)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
